# 02 - Transforms Usage

This notebook is a practical guide to Nebula Space Toolkit's `nstk.transforms` module.

It walks through:
- geodetic, ECEF, ENU, and AER conversions (scalar + vectorized APIs)
- coarse ECI/ECEF/geodetic transforms (position and velocity)
- timed Orekit-backed frame transforms via `transform(...)`
- core constants and function naming patterns


## Conventions And Naming

- Angular inputs are generally **radians** unless a function ends in `_deg`.
- Distances are meters, velocities are m/s, accelerations are m/s^2.
- Function suffix patterns:
  - scalar function: one point/state
  - `_vec_xyz` / `_vec_llh` / `_vec_enu`: parallel 1D-array inputs
  - `_vec_ecef` / `_vec_lla` / `_vec_enu3` / `_vec_aer3`: `(N,3)` array style


In [ ]:
# Ensure local package import when running from this examples/ folder.
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_repo_root = _cwd if (_cwd / "nstk").is_dir() else _cwd.parent
if (_repo_root / "nstk").is_dir() and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [ ]:
from time import perf_counter

import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.utils import iers

import nstk.transforms as tf
from nstk.time_utils import astropy_time_to_orekit_date

np.set_printoptions(precision=6, suppress=True)

# Avoid long network waits when UT1/IERS tables are requested offline.
iers.conf.auto_download = False
iers.conf.auto_max_age = None


## Public API Map

This cell lists every callable exported by `nstk.transforms`.


In [ ]:
callable_exports = sorted([name for name in tf.__all__ if callable(getattr(tf, name))])

categories = {
    "Geodetic/ECEF": [n for n in callable_exports if "geodetic" in n or "ecef" in n and "coarse" not in n and n != "transform"],
    "ENU": [n for n in callable_exports if "enu" in n and "coarse" not in n],
    "AER": [n for n in callable_exports if "aer" in n],
    "Coarse ECI/ECEF": [n for n in callable_exports if n.startswith("coarse_")],
    "Timed Frame": [n for n in callable_exports if n == "transform"],
}

print(f"Total callable exports: {len(callable_exports)}")
for k, v in categories.items():
    unique = sorted(set(v))
    print(f"\n{k} ({len(unique)}):")
    print(", ".join(unique))


## Reference Scenario

Create a single observer and a small set of target points that we reuse in all transform families.


In [ ]:
def wrap_pi(rad):
    return (rad + np.pi) % (2.0 * np.pi) - np.pi


obs_lat_deg = 34.0
obs_lon_deg = -118.0
obs_h_m = 250.0

obs_lat = np.deg2rad(obs_lat_deg)
obs_lon = np.deg2rad(obs_lon_deg)

# Targets near the observer.
targets_deg_m = np.array(
    [
        [34.0500, -117.8500, 700.0],
        [34.0200, -118.0200, 1200.0],
        [33.9800, -117.9000, 50.0],
        [34.1200, -118.1500, 3000.0],
    ],
    dtype=np.float64,
)

lat_vec = np.deg2rad(targets_deg_m[:, 0])
lon_vec = np.deg2rad(targets_deg_m[:, 1])
h_vec = targets_deg_m[:, 2]

print("Observer [deg,deg,m]:", obs_lat_deg, obs_lon_deg, obs_h_m)
print("Targets shape:", targets_deg_m.shape)


## 1) Geodetic <-> ECEF (Scalar)


In [ ]:
lat = lat_vec[0]
lon = lon_vec[0]
h_m = float(h_vec[0])

x_m, y_m, z_m = tf.geodetic2ecef(lat, lon, h_m)
lat_back, lon_back, h_back = tf.ecef2geodetic(x_m, y_m, z_m)
lat_back_deg, lon_back_deg, h_back_deg = tf.ecef2geodetic_deg(x_m, y_m, z_m)

print("ECEF [m]:", x_m, y_m, z_m)
print("roundtrip lat err [rad]:", float(lat_back - lat))
print("roundtrip lon err [rad]:", float(wrap_pi(lon_back - lon)))
print("roundtrip h err [m]:", float(h_back - h_m))
print("roundtrip [deg,deg,m]:", lat_back_deg, lon_back_deg, h_back_deg)


## 2) Geodetic <-> ECEF (Vectorized Variants)

Demonstrates all vectorized forms in this family:
- `geodetic2ecef_vec_llh`
- `geodetic2ecef_vec_lla`
- `ecef2geodetic_vec_xyz`
- `ecef2geodetic_vec_ecef`
- `ecef2geodetic_vec_ecef_deg`


In [ ]:
x_arr, y_arr, z_arr = tf.geodetic2ecef_vec_llh(lat_vec, lon_vec, h_vec)
r_ecef_from_llh = np.column_stack((x_arr, y_arr, z_arr))

lla_rad_m = np.column_stack((lat_vec, lon_vec, h_vec))
r_ecef_from_lla = tf.geodetic2ecef_vec_lla(lla_rad_m)

lat_xyz, lon_xyz, h_xyz = tf.ecef2geodetic_vec_xyz(x_arr, y_arr, z_arr)
lat_ecef, lon_ecef, h_ecef = tf.ecef2geodetic_vec_ecef(r_ecef_from_llh)
lat_deg, lon_deg, h_deg = tf.ecef2geodetic_vec_ecef_deg(r_ecef_from_llh)

print("ecef_vec_llh vs ecef_vec_lla allclose:", bool(np.allclose(r_ecef_from_llh, r_ecef_from_lla)))
print("vec_xyz vs vec_ecef lat allclose:", bool(np.allclose(lat_xyz, lat_ecef)))
print("vec_xyz vs input max |lat err| [rad]:", float(np.max(np.abs(lat_xyz - lat_vec))))
print("vec_xyz vs input max |lon err| [rad]:", float(np.max(np.abs(wrap_pi(lon_xyz - lon_vec)))))
print("vec_xyz vs input max |h err| [m]:", float(np.max(np.abs(h_xyz - h_vec))))
print("first row ecef2geodetic_vec_ecef_deg [deg,deg,m]:", float(lat_deg[0]), float(lon_deg[0]), float(h_deg[0]))


## 3) ENU Basis, Scalar ENU/ECEF/Geodetic, And Delta Rotations

This section uses:
- `enu_basis_from_latlon`, `enu_basis_from_ecef_xyz`
- `geodetic2enu`, `ecef2enu`, `enu2ecef`, `enu2geodetic`
- `ecef2enu_delta`, `enu2ecef_delta`


In [ ]:
# Pick first target as scalar example.
lat_t = float(lat_vec[0])
lon_t = float(lon_vec[0])
h_t = float(h_vec[0])

e_m, n_m, u_m = tf.geodetic2enu(lat_t, lon_t, h_t, obs_lat, obs_lon, obs_h_m)
x_t, y_t, z_t = tf.enu2ecef(e_m, n_m, u_m, obs_lat, obs_lon, obs_h_m)
e_chk, n_chk, u_chk = tf.ecef2enu(x_t, y_t, z_t, obs_lat, obs_lon, obs_h_m)

lat_rt, lon_rt, h_rt = tf.enu2geodetic(e_m, n_m, u_m, obs_lat, obs_lon, obs_h_m)

dx_m, dy_m, dz_m = tf.enu2ecef_delta(e_m, n_m, u_m, obs_lat, obs_lon)
e_d, n_d, u_d = tf.ecef2enu_delta(dx_m, dy_m, dz_m, obs_lat, obs_lon)

basis_ll = tf.enu_basis_from_latlon(obs_lat, obs_lon)
x_obs, y_obs, z_obs = tf.geodetic2ecef(obs_lat, obs_lon, obs_h_m)
basis_xyz = tf.enu_basis_from_ecef_xyz(x_obs, y_obs, z_obs)

print("ENU [m]:", e_m, n_m, u_m)
print("ECEF->ENU backcheck max abs diff [m]:", float(np.max(np.abs([e_chk - e_m, n_chk - n_m, u_chk - u_m]))))
print("ENU->geodetic max errors [rad,rad,m]:", float(abs(lat_rt - lat_t)), float(abs(wrap_pi(lon_rt - lon_t))), float(abs(h_rt - h_t)))
print("delta roundtrip max abs diff [m]:", float(np.max(np.abs([e_d - e_m, n_d - n_m, u_d - u_m]))))
print("basis(latlon) orthonormal check ||R^T R - I||:", float(np.linalg.norm(basis_ll.T @ basis_ll - np.eye(3))))
print("basis(ecef xyz) orthonormal check ||R^T R - I||:", float(np.linalg.norm(basis_xyz.T @ basis_xyz - np.eye(3))))


## 4) ENU Vectorized APIs

Demonstrates all vectorized ENU path variants:
- `geodetic2enu_vec_llh`, `geodetic2enu_vec_lla`
- `ecef2enu_vec_xyz`, `ecef2enu_vec_ecef`
- `enu2ecef_vec_enu`, `enu2ecef_vec_enu3`
- `enu2geodetic_vec_enu`, `enu2geodetic_vec_enu3`


In [ ]:
e_llh, n_llh, u_llh = tf.geodetic2enu_vec_llh(lat_vec, lon_vec, h_vec, obs_lat, obs_lon, obs_h_m)
enu_stack = np.column_stack((e_llh, n_llh, u_llh))

enu_from_lla = tf.geodetic2enu_vec_lla(np.column_stack((lat_vec, lon_vec, h_vec)), obs_lat, obs_lon, obs_h_m)

x_from_enu, y_from_enu, z_from_enu = tf.enu2ecef_vec_enu(e_llh, n_llh, u_llh, obs_lat, obs_lon, obs_h_m)
r_ecef_from_enu = np.column_stack((x_from_enu, y_from_enu, z_from_enu))
r_ecef_from_enu3 = tf.enu2ecef_vec_enu3(enu_stack, obs_lat, obs_lon, obs_h_m)

e_xyz, n_xyz, u_xyz = tf.ecef2enu_vec_xyz(x_from_enu, y_from_enu, z_from_enu, obs_lat, obs_lon, obs_h_m)
e_ecef, n_ecef, u_ecef = tf.ecef2enu_vec_ecef(r_ecef_from_enu, obs_lat, obs_lon, obs_h_m)

lat_enu, lon_enu, h_enu = tf.enu2geodetic_vec_enu(e_llh, n_llh, u_llh, obs_lat, obs_lon, obs_h_m)
lat_enu3, lon_enu3, h_enu3 = tf.enu2geodetic_vec_enu3(enu_stack, obs_lat, obs_lon, obs_h_m)

print("geodetic2enu_vec_llh vs vec_lla allclose:", bool(np.allclose(enu_stack, enu_from_lla)))
print("enu2ecef_vec_enu vs vec_enu3 allclose:", bool(np.allclose(r_ecef_from_enu, r_ecef_from_enu3)))
print("ecef2enu_vec_xyz vs vec_ecef allclose:", bool(np.allclose(np.column_stack((e_xyz, n_xyz, u_xyz)), np.column_stack((e_ecef, n_ecef, u_ecef)))))
print("enu2geodetic_vec_enu vs vec_enu3 max |h diff| [m]:", float(np.max(np.abs(h_enu - h_enu3))))
print("enu2geodetic max |lat err| [rad]:", float(np.max(np.abs(lat_enu - lat_vec))))
print("enu2geodetic max |lon err| [rad]:", float(np.max(np.abs(wrap_pi(lon_enu - lon_vec)))))


## 5) AER Scalar Workflows

Demonstrates scalar AER path functions:
- `enu2aer`, `geodetic2aer`, `ecef2aer`
- `aer2enu`, `aer2ecef`, `aer2geodetic`


In [ ]:
az_g, el_g, sr_g = tf.geodetic2aer(lat_t, lon_t, h_t, obs_lat, obs_lon, obs_h_m)
az_e, el_e, sr_e = tf.ecef2aer(x_t, y_t, z_t, obs_lat, obs_lon, obs_h_m)
az_enu, el_enu, sr_enu = tf.enu2aer(e_m, n_m, u_m)

e_from_aer, n_from_aer, u_from_aer = tf.aer2enu(az_g, el_g, sr_g)
x_from_aer, y_from_aer, z_from_aer = tf.aer2ecef(az_g, el_g, sr_g, obs_lat, obs_lon, obs_h_m)
lat_from_aer, lon_from_aer, h_from_aer = tf.aer2geodetic(az_g, el_g, sr_g, obs_lat, obs_lon, obs_h_m)

print("geodetic2aer vs ecef2aer [az,el,sr] close:", bool(np.allclose([az_g, el_g, sr_g], [az_e, el_e, sr_e])))
print("geodetic2aer vs enu2aer [az,el,sr] close:", bool(np.allclose([az_g, el_g, sr_g], [az_enu, el_enu, sr_enu])))
print("aer2enu backcheck max abs diff [m]:", float(np.max(np.abs([e_from_aer - e_m, n_from_aer - n_m, u_from_aer - u_m]))))
print("aer2ecef backcheck max abs diff [m]:", float(np.max(np.abs([x_from_aer - x_t, y_from_aer - y_t, z_from_aer - z_t]))))
print("aer2geodetic max errors [rad,rad,m]:", float(abs(lat_from_aer - lat_t)), float(abs(wrap_pi(lon_from_aer - lon_t))), float(abs(h_from_aer - h_t)))


## 6) AER Vectorized APIs

Demonstrates vectorized AER family:
- `geodetic2aer_vec_llh`
- `ecef2aer_vec_xyz`
- `aer2ecef_vec_aer`, `aer2ecef_vec_aer3`
- `aer2geodetic_vec_aer`, `aer2geodetic_vec_aer3`


In [ ]:
az_vec, el_vec, sr_vec = tf.geodetic2aer_vec_llh(lat_vec, lon_vec, h_vec, obs_lat, obs_lon, obs_h_m)

x_a, y_a, z_a = tf.aer2ecef_vec_aer(az_vec, el_vec, sr_vec, obs_lat, obs_lon, obs_h_m)
r_ecef_aer = np.column_stack((x_a, y_a, z_a))
r_ecef_aer3 = tf.aer2ecef_vec_aer3(np.column_stack((az_vec, el_vec, sr_vec)), obs_lat, obs_lon, obs_h_m)

az_xyz, el_xyz, sr_xyz = tf.ecef2aer_vec_xyz(x_a, y_a, z_a, obs_lat, obs_lon, obs_h_m)

lat_a, lon_a, h_a = tf.aer2geodetic_vec_aer(az_vec, el_vec, sr_vec, obs_lat, obs_lon, obs_h_m)
lat_a3, lon_a3, h_a3 = tf.aer2geodetic_vec_aer3(np.column_stack((az_vec, el_vec, sr_vec)), obs_lat, obs_lon, obs_h_m)

print("aer2ecef_vec_aer vs vec_aer3 allclose:", bool(np.allclose(r_ecef_aer, r_ecef_aer3)))
print("geodetic2aer_vec_llh vs ecef2aer_vec_xyz allclose:", bool(np.allclose(np.column_stack((az_vec, el_vec, sr_vec)), np.column_stack((az_xyz, el_xyz, sr_xyz)))))
print("aer2geodetic_vec_aer vs vec_aer3 max |h diff| [m]:", float(np.max(np.abs(h_a - h_a3))))
print("aer2geodetic max |lat err| [rad]:", float(np.max(np.abs(lat_a - lat_vec))))
print("aer2geodetic max |lon err| [rad]:", float(np.max(np.abs(wrap_pi(lon_a - lon_vec)))))


## 7) Coarse ECI <-> ECEF And Coarse ECI -> Geodetic

This section exercises all coarse-transform functions:
- scalar position and position/velocity variants
- vectorized variants and alias helpers
- geodetic outputs in radians and degrees


In [ ]:
epoch = Time("2026-01-01T00:00:00", scale="utc")
dt_s = np.arange(0.0, 600.0, 60.0, dtype=np.float64)
times = epoch + dt_s * u.s

radius_m = 7000e3
omega = 2.0 * np.pi / (95.0 * 60.0)
theta = omega * dt_s

r_eci = np.column_stack(
    (
        radius_m * np.cos(theta),
        radius_m * np.sin(theta),
        1.0e5 * np.sin(0.5 * theta),
    )
).astype(np.float64)

v_eci = np.column_stack(
    (
        -radius_m * omega * np.sin(theta),
        radius_m * omega * np.cos(theta),
        1.0e5 * 0.5 * omega * np.cos(0.5 * theta),
    )
).astype(np.float64)

jd_ut1 = times.ut1.jd.astype(np.float64)
jd_tt = times.tt.jd.astype(np.float64)

# Scalar calls
x_ecef0, y_ecef0, z_ecef0 = tf.coarse_eci2ecef_pos(*r_eci[0], jd_ut1[0], jd_tt[0])
x_eci0, y_eci0, z_eci0 = tf.coarse_ecef2eci_pos(x_ecef0, y_ecef0, z_ecef0, jd_ut1[0], jd_tt[0])
x_eci0_alias, y_eci0_alias, z_eci0_alias = tf.coarse_ecef2eci(x_ecef0, y_ecef0, z_ecef0, jd_ut1[0], jd_tt[0])

pv_ecef0 = tf.coarse_eci2ecef_pos_vel(*r_eci[0], *v_eci[0], jd_ut1[0], jd_tt[0])
pv_eci0 = tf.coarse_ecef2eci_pos_vel(*pv_ecef0, jd_ut1[0], jd_tt[0])

lat0_rad, lon0_rad, h0_m = tf.coarse_eci2geodetic(*r_eci[0], jd_ut1[0], jd_tt[0])
lat0_deg, lon0_deg, h0_deg_m = tf.coarse_eci2geodetic_deg(*r_eci[0], jd_ut1[0], jd_tt[0])

# Vector calls
r_ecef = tf.coarse_eci2ecef_pos_vec(r_eci, jd_ut1, jd_tt)
r_eci_back = tf.coarse_ecef2eci_pos_vec(r_ecef, jd_ut1, jd_tt)
r_eci_back_alias = tf.coarse_ecef2eci_vec(r_ecef, jd_ut1, jd_tt)

r_ecef_pv, v_ecef_pv = tf.coarse_eci2ecef_pos_vel_vec(r_eci, v_eci, jd_ut1, jd_tt)
r_eci_pv_back, v_eci_pv_back = tf.coarse_ecef2eci_pos_vel_vec(r_ecef_pv, v_ecef_pv, jd_ut1, jd_tt)

lat_rad_vec, lon_rad_vec, h_m_vec = tf.coarse_eci2geodetic_vec(r_eci, jd_ut1, jd_tt)
lat_deg_vec, lon_deg_vec, h_deg_vec = tf.coarse_eci2geodetic_vec_deg(r_eci, jd_ut1, jd_tt)

err_pos = np.linalg.norm(r_eci_back - r_eci, axis=1)
err_pos_alias = np.linalg.norm(r_eci_back_alias - r_eci, axis=1)
err_pos_pv = np.linalg.norm(r_eci_pv_back - r_eci, axis=1)
err_vel_pv = np.linalg.norm(v_eci_pv_back - v_eci, axis=1)

print("Scalar coarse_ecef2eci alias exact match:", bool(np.allclose([x_eci0, y_eci0, z_eci0], [x_eci0_alias, y_eci0_alias, z_eci0_alias])))
print("Vector position roundtrip max error [m]:", float(np.max(err_pos)))
print("Vector position roundtrip alias max error [m]:", float(np.max(err_pos_alias)))
print("Vector pos+vel roundtrip max position error [m]:", float(np.max(err_pos_pv)))
print("Vector pos+vel roundtrip max velocity error [m/s]:", float(np.max(err_vel_pv)))
print("First coarse geodetic sample [rad,rad,m]:", float(lat0_rad), float(lon0_rad), float(h0_m))
print("First coarse geodetic sample [deg,deg,m]:", float(lat0_deg), float(lon0_deg), float(h0_deg_m))
print("coarse_eci2geodetic_vec first row [deg,deg,m]:", float(lat_deg_vec[0]), float(lon_deg_vec[0]), float(h_deg_vec[0]))


## 8) Timed Orekit-Backed Frame Transform (`transform`)

`transform(...)` handles arbitrary frame transforms with time dependence.

Notes:
- `time` accepts `astropy.time.Time`, Orekit `AbsoluteDate`, unix-second numerics, or time `Quantity`.
- For numeric inputs, values are interpreted as **unix seconds**.
- If you pass `Quantity` vectors in, outputs preserve quantity units.


In [ ]:

times_tf = epoch + np.array([0.0, 45.0, 90.0], dtype=np.float64) * u.s

r_gcrf = r_eci[:3].copy()
v_gcrf = v_eci[:3].copy()
a_gcrf = np.zeros_like(r_gcrf)

# Position-only transform.
p_itrf_only, _, _ = tf.transform(
    from_frame="gcrf",
    to_frame="itrf",
    time=times_tf,
    position=r_gcrf,
)

# Position + velocity transform and roundtrip.
p_itrf, v_itrf, _ = tf.transform(
    from_frame="gcrf",
    to_frame="itrf",
    time=times_tf,
    position=r_gcrf,
    velocity=v_gcrf,
)
p_back, v_back, _ = tf.transform("itrf", "gcrf", times_tf, p_itrf, velocity=v_itrf)

# Alias frames ("eci"->"gcrf", "ecef"->"itrf").
p_alias, _, _ = tf.transform("eci", "ecef", times_tf, r_gcrf)

# Quantity-preserving path with acceleration.
p_q, v_q, a_q = tf.transform(
    from_frame="gcrf",
    to_frame="itrf",
    time=times_tf,
    position=r_gcrf * u.m,
    velocity=v_gcrf * (u.m / u.s),
    acceleration=a_gcrf * (u.m / (u.s**2)),
)

# Acceleration-only mode: velocity output stays None.
_, v_none, a_only = tf.transform(
    from_frame="gcrf",
    to_frame="itrf",
    time=times_tf,
    position=r_gcrf,
    acceleration=a_gcrf,
)

# Equivalent time forms: astropy Time, unix seconds, Orekit AbsoluteDate list.
unix_times = times_tf.utc.unix.astype(np.float64)
p_unix, _, _ = tf.transform("gcrf", "itrf", unix_times, r_gcrf)

t0_abs = astropy_time_to_orekit_date(times_tf[0])
abs_times = [t0_abs.shiftedBy(0.0), t0_abs.shiftedBy(45.0), t0_abs.shiftedBy(90.0)]
p_abs, _, _ = tf.transform("gcrf", "itrf", abs_times, r_gcrf)

print("position-only output shape:", p_itrf_only.shape)
print("PV output shapes:", p_itrf.shape, v_itrf.shape)
print("gcrf->itrf->gcrf max position error [m]:", float(np.max(np.linalg.norm(p_back - r_gcrf, axis=1))))
print("gcrf->itrf->gcrf max velocity error [m/s]:", float(np.max(np.linalg.norm(v_back - v_gcrf, axis=1))))
print("alias frames match canonical names:", bool(np.allclose(p_alias, p_itrf_only)))
print("Quantity output types:", type(p_q).__name__, type(v_q).__name__, type(a_q).__name__)
print("acceleration-only returns velocity None:", v_none is None)
print("time forms consistent (Time vs unix):", bool(np.allclose(p_itrf_only, p_unix)))
print("time forms consistent (Time vs AbsoluteDate):", bool(np.allclose(p_itrf_only, p_abs)))


## 9) Constants Cheat Sheet


In [ ]:
print("WGS84_A [m]:", tf.WGS84_A)
print("WGS84_B [m]:", tf.WGS84_B)
print("WGS84_E2:", tf.WGS84_E2)
print("DEG2RAD / RAD2DEG:", tf.DEG2RAD, tf.RAD2DEG)
print("J2000_JD:", tf.J2000_JD)
print("EARTH_OMEGA [rad/s]:", tf.EARTH_OMEGA)
print("DAS2R [rad/arcsec]:", tf.DAS2R)


Next notebook: **03 - Walker Constellation**.